# 第 4 章 · GitHub、自动检查、合并

**为什么要做**：代码只在自己电脑上，硬盘坏了就没了，别人也看不到；第 3 章的检查也只装在你这台电脑上。
公司里的做法：正本放在 GitHub；每个改动，GitHub 自动再检查一遍；绿了、有人看过，才能并入正本。

**做完得到什么**：
- 正本在 GitHub 上（私有）
- 每个改动都有自动检查的结果：绿勾或红叉
- 改动要你点"合并"才算数

下面三格只是查看。从上往下一格一格点 ▶。

In [1]:
# 进到项目目录（这格不用看懂）
import os
import subprocess

os.chdir(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
os.environ.update(NO_COLOR="1", TERM="dumb", GIT_PAGER="cat", PAGER="cat",
                  GH_SPINNER_DISABLED="1", GH_PAGER="cat")
print("已进入项目目录")

已进入项目目录


## 1 · 正本在哪

**为什么**：正本放在服务器上，本地只是工作副本。电脑坏了，正本还在。

In [2]:
!gh repo view --json nameWithOwner,visibility --jq '.nameWithOwner + "   " + .visibility'

pipi0616/rare-disease-agent   PRIVATE


👀 我们的项目在 GitHub 上，PRIVATE 表示私有，只有你能看。

## 2 · 每个改动都是一个"合并请求"

**为什么**：改动不直接进正本，而是先提一个**合并请求（pull request，简称 PR）**："我改了这些，请检查后并入"。

In [3]:
!gh pr list --state all --json number,state,title --jq '.[] | "#\(.number)   \(.state)   \(.title)"'

#2   OPEN   feat(data): Phenopacket Store loader with checksum-verified fetch and PMID split
#1   MERGED   ci: run the checks on GitHub for every pull request


👀 每一行是一个改动。MERGED = 已经并入正本，OPEN = 还在等你点合并。

## 3 · 自动检查拦住过一次问题

**为什么**：GitHub 每收到一个改动，就在一台干净的电脑上把检查全部重跑一遍。不管是谁、从哪台电脑交的，都要过这一关。

In [4]:
!gh run list --status failure --json createdAt,displayTitle --jq 'reverse | .[] | .createdAt[:16] + "   " + .displayTitle'

Working...


2026-10-07T09:07   ci: run the checks on GitHub for every pull request
2026-10-07T09:59   feat(data): Phenopacket Store loader with checksum-verified fetch and PMID split


👀 这是所有被拦下的记录（每行是时间和那个改动的标题）。两次都是真实发生的：
- **第一次**：检查的设置里写错了一个版本号。
- **第二次**：有两个代码文件没被存进去——"别记"清单写得太宽，把它们也挡住了。你的电脑上这两个文件是在的，
  所以本地什么都发现不了；GitHub 在一台干净的电脑上一跑，立刻报"找不到 raredx.data"。

两次都在进入正本之前被拦住了。第二次正好说明为什么要在干净的电脑上再查一遍：**"在我电脑上能跑"不算数**。
除了这两次，其余每一次都是 success。

## 要记住的

| | 是什么 | 公司为什么要它 |
|---|---|---|
| GitHub | 放正本的地方 | 不怕丢，大家都能看 |
| 合并请求 PR | "我改了这些，请检查后并入" | 每个改动都经过检查和审阅 |
| 自动检查 CI（GitHub Actions） | GitHub 自动跑检查 | 不管谁交的，都检查一遍 |

**面试时可以这么说**："所有改动走 PR，GitHub Actions 跑 CI，绿了才合并。"